# Predicting US Flight Delays

Dataset: USDOT 2015 Flight Delays (from Kaggle, loaded from HDFS)
Target: is_delayed = ARRIVAL_DELAY > 15 minutes (yes or no)

This notebook follows these steps:
1. Load the data
2. Look at the data & parse types (EDA & Parsing)
3. Clean the data
4. Feature Engineering (turn text into numbers)
5. Train X models
6. Save the results for Streamlit

## **1) Start Spark and load the file from HDFS**
Spark is the tool we use to read and process the data. We initialize our `SparkSession` and read the file directly from HDFS.

In [ ]:
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.appName("flight-delays")
         .config("spark.driver.memory", "4g").getOrCreate())
spark


CSV reading options configured (header, delimiter)

In [ ]:
raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "false")  
    .option("delimiter", ",")
    .csv("hdfs://localhost:9000/data/raw/flights.csv")
)

print("Rows:", raw.count())
print("Columns:", len(raw.columns))

**Result:** We have 5,819,079 rows and 31 columns. This matches the number Kaggle
announces for this dataset, so the file loaded correctly.

## **2) Parsing - Inspect and cast data types**

In [ ]:
import pandas as pd
raw.printSchema() 
pd.set_option("display.max_columns", None)
raw.limit(5).toPandas()

**Column names validation:** Columns seem to use use short, clean names without spaces.

In [ ]:
first_non_null = raw.select([F.first(c, ignorenulls=True).alias(c) for c in raw.columns]).first().asDict()
pd.Series(first_non_null, name="example_value")

### **2.1) Casting**
After inspection, every column (even numbers) is shown as **string**, this is because Spark loads everything as text by default to avoid a costly double read of the CSV file. 
 

We can confirm that we have:
* **5 text columns:** `AIRLINE`, `TAIL_NUMBER`, `ORIGIN_AIRPORT`, `DESTINATION_AIRPORT`, `CANCELLATION_REASON`.
* **26 numeric columns:** `YEAR`, `MONTH`, `DAY`, `DAY_OF_WEEK`, `FLIGHT_NUMBER`, `SCHEDULED_DEPARTURE`, `DEPARTURE_TIME`, `DEPARTURE_DELAY`, `TAXI_OUT`, `WHEELS_OFF`, `SCHEDULED_TIME`, `ELAPSED_TIME`, `AIR_TIME`, `DISTANCE`, `WHEELS_ON`, `TAXI_IN`, `SCHEDULED_ARRIVAL`, `ARRIVAL_TIME`, `ARRIVAL_DELAY`, `DIVERTED`, `CANCELLED`, `AIR_SYSTEM_DELAY`, `SECURITY_DELAY`, `AIRLINE_DELAY`, `LATE_AIRCRAFT_DELAY`, `WEATHER_DELAY`.

We cast all 26 numeric columns to `double` and verify the updated schema

In [ ]:
numerical_cols = [
    "YEAR", "MONTH", "DAY", "DAY_OF_WEEK", "FLIGHT_NUMBER",
    "SCHEDULED_DEPARTURE", "DEPARTURE_TIME", "DEPARTURE_DELAY",
    "TAXI_OUT", "WHEELS_OFF", "SCHEDULED_TIME", "ELAPSED_TIME",
    "AIR_TIME", "DISTANCE", "WHEELS_ON", "TAXI_IN",
    "SCHEDULED_ARRIVAL", "ARRIVAL_TIME", "ARRIVAL_DELAY",
    "DIVERTED", "CANCELLED", "AIR_SYSTEM_DELAY", "SECURITY_DELAY",
    "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"
]

df_parsed = raw
for c in numerical_cols:
    df_parsed = df_parsed.withColumn(c, F.col(c).cast("double"))

df_parsed.printSchema()

## **3) Smaller Sample for Exploration**

The full file has 5.8 million rows. We take a random 5% sample just for this exploring part. We will use the full file later, when we train the final model.

In [ ]:
eda_sample = raw.sample(fraction=0.05, seed=42).cache()
print("Sample size:", eda_sample.count())

We have approximately 290k lines to use for data exploration

### **3.1) Check for Missing Values**
We count how many values are missing in each column.

In [ ]:
import pandas as pd

missing_col_values = [F.sum(F.isnull(c).cast("int")).alias(c) for c in eda_sample.columns]
counts = eda_sample.select(missing_col_values).first().asDict()

pd.Series(counts, name="missing").sort_values(ascending=False)

What we notice so far:
1. **Pre-departure features are 100% complete:** MONTH, DAY_OF_WEEK, AIRLINE, ORIGIN_AIRPORT, DESTINATION_AIRPORT, SCHEDULED_DEPARTURE, SCHEDULED_TIME, and DISTANCE all have 0 missing values.

2. **Delay causes are structural/conditional:** The 5 delay-cause columns (WEATHER_DELAY, AIRLINE_DELAY, LATE_AIRCRAFT_DELAY, AIR_SYSTEM_DELAY, SECURITY_DELAY) have 237,966 missing values because they are only filled in when a flight is actually delayed.

3. **Missing arrival data is tied to cancellations/diversions**: ARRIVAL_DELAY, ELAPSED_TIME, and AIR_TIME are missing for 5,234 flights


In [ ]:
eda_sample.groupBy("CANCELLED", "DIVERTED").agg(
    F.count("*").alias("total_flights"),
    F.sum(F.isnull("ARRIVAL_DELAY").cast("int")).alias("missing_arrival_delay")
).show()


This confirms that every single missing arrival delay belongs to a cancelled or diverted flight.

### **3.2) Distinct values per categorical column (full dataset)**
Next, we check how many unique values each text column has in our full `df_parsed` DataFrame.

In [ ]:
cat_cols = ["AIRLINE", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "CANCELLATION_REASON"]

distinct_counts = {
    c: df_parsed.select(F.countDistinct(F.col(c))).first()[0]
    for c in cat_cols
}

pd.Series(distinct_counts, name="distinct_values").sort_values(ascending=False)

Across the full dataset (`df_parsed`), `TAIL_NUMBER` has 4,897 distinct values and the airport columns have 629 / 628 distinct values (which includes non-standard numeric codes alongside regular 3-letter airport codes before cleaning). By contrast, `AIRLINE` has 14 distinct values and `CANCELLATION_REASON` has 4. 
This tells us upfront that columns like airports and tail numbers will need careful handling during feature engineering.

### **3.3) Target distribution (ARRIVAL_DELAY > 15)**
Our goal is to predict whether a flight arrives more than 15 minutes late (`is_delayed = 1` vs `0`), we check the distribution of the column in our sample dataset.

In [ ]:
import matplotlib.pyplot as plt

valid_flights = (
    eda_sample
    .filter(F.col("ARRIVAL_DELAY").isNotNull())
    .withColumn("is_delayed", (F.col("ARRIVAL_DELAY") > 15).cast("int"))
)

counts = valid_flights.groupBy("is_delayed").count().orderBy("is_delayed").toPandas()
labels = [f"{'Delayed >15m' if r.is_delayed else 'Not delayed'}\n({r['count']:,})" for _, r in counts.iterrows()]

plt.figure(figsize=(5, 5))
plt.pie(counts["count"], labels=labels, autopct="%.1f%%", startangle=90)
plt.title("Target distribution (is_delayed)")
plt.show()

This shows a clear class imbalance where **82.2%** are not delayed (`234,717`) and **17.8%** are delayed (`50,825`). 


### **3.4) Correlations with the target**
Next, we check linear correlations between numeric columns and `is_delayed` on flights with a non-null `ARRIVAL_DELAY`, to spot which numeric features move with delays and which columns happen after takeoff.

In [ ]:
import matplotlib.pyplot as plt

corr_series = pd.Series(correlations).sort_values()

plt.figure(figsize=(7, 4))
plt.barh(corr_series.index, corr_series.values)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Correlation with is_delayed")
plt.title("Numeric correlation with arrival delay (> 15 min)")
plt.tight_layout()
plt.show()

Looking at the chart, `DEPARTURE_DELAY` (`0.60`) and `TAXI_OUT` (`0.25`) have the strongest positive link to arriving more than 15 minutes late, while `SCHEDULED_DEPARTURE` (`0.14`) and `TAXI_IN` (`0.11`) have smaller positive links, and the rest sit fairly close to zero.

### Staistical findings:
Three main facts will guide how we clean the data and build our model:
1. **Missing values aren't random:** `ARRIVAL_DELAY` is missing on cancelled/diverted flights, and delay-cause fields only show up on delayed flights—so we drop cancelled/diverted flights and keep only relevant columns.
2. **Unuseful columns and class imbalance:** `DEPARTURE_DELAY` and `TAXI_OUT` happen after/during takeoff, so they can't be used for pre-departure predictions. Also, most flights aren't delayed (~82% vs ~18%), so accuracy alone isn't enough later.
3. **Unique text values need cleanup:** Across the full dataset, `TAIL_NUMBER` has 4,897 unique values and the airport columns have 629 / 628 (including non-standard numeric codes), compared to 14 airlines and 4 cancellation reasons—so we'll clean odd codes and group rare values carefully.

## **4) Clean the data**
### **4.1) Missing values + incomplete flights**
Now we move to the full dataset (`df_parsed`) to build a clean table with no missing values. 
We'll also remove cancelled or diverted flights (since they don't have normal arrival delays) along with any rows missing key fields like `ARRIVAL_DELAY`, `SCHEDULED_TIME`, `DISTANCE`, or `SCHEDULED_DEPARTURE`, leaving only complete flights we can actually use.

In [ ]:
cols = [
    "MONTH", "DAY", "DAY_OF_WEEK", "AIRLINE",
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
    "SCHEDULED_DEPARTURE", "SCHEDULED_TIME", "DISTANCE",
    "ARRIVAL_DELAY", "CANCELLED", "DIVERTED"
]

df_clean = df_parsed.select(cols)
n_start = df_clean.count()
print(f"Start rows: {n_start:,}")

df_clean = (
    df_clean
    .filter((F.col("CANCELLED") == 0) & (F.col("DIVERTED") == 0))
    .dropna(subset=[
        "MONTH", "DAY", "DAY_OF_WEEK", "AIRLINE",
        "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
        "SCHEDULED_DEPARTURE", "SCHEDULED_TIME", "DISTANCE",
        "ARRIVAL_DELAY"
    ])
)

n_after_missing = df_clean.count()
print(f"After missing & cancelled/diverted drop: {n_after_missing:,} (dropped {n_start - n_after_missing:,})")

### **4.2) Removing duplicate rows**
Next, we drop exact duplicate rows with `dropDuplicates()` so repeated flights don't bias training, and check the row count before and after.

In [ ]:
before_dedup = df_clean.count()
df_clean = df_clean.dropDuplicates()
after_dedup = df_clean.count()

print(f"After dropDuplicates: {after_dedup:,} (dropped {before_dedup - after_dedup:,})")

### **4.3) Filtering impossible values and outliers**
Here we remove non-standard numeric airport codes, require positive scheduled times and distances, and keep `ARRIVAL_DELAY` within `[-120, 1500]` minutes to filter extreme outliers while logging the row count before and after.

In [ ]:
# Step 1 inside 4.3: quick check on numeric airport codes and extreme delay bounds
num_origins = df_clean.filter(F.col("ORIGIN_AIRPORT").rlike("^[0-9]+$")).count()
num_dests = df_clean.filter(F.col("DESTINATION_AIRPORT").rlike("^[0-9]+$")).count()
print(f"Numeric airport rows -> origin: {num_origins:,}, dest: {num_dests:,}")

df_clean.select(
    F.min("ARRIVAL_DELAY").alias("min_delay"),
    F.max("ARRIVAL_DELAY").alias("max_delay"),
    F.min("SCHEDULED_TIME").alias("min_sched_time"),
    F.min("DISTANCE").alias("min_dist")
).show()

# Step 2 inside 4.3: filter and log row count before and after
before_outliers = df_clean.count()

df_clean = (
    df_clean
    .filter(~F.col("ORIGIN_AIRPORT").rlike("^[0-9]+$") & ~F.col("DESTINATION_AIRPORT").rlike("^[0-9]+$"))
    .filter(
        (F.col("SCHEDULED_TIME") > 0) &
        (F.col("DISTANCE") > 0) &
        (F.col("ARRIVAL_DELAY").between(-120, 1500))
    )
)

after_outliers = df_clean.count()
print(f"After impossible/outlier filter: {after_outliers:,} (dropped {before_outliers - after_outliers:,})")

Based on what we just checked, we see delays reaching up to `1,971` minutes while scheduled times (`18` min minimum) and distances (`31` mi minimum) are already positive. 
Filtering out numeric airport codes and extreme `ARRIVAL_DELAY` values outside `[-120, 1500]` minutes removed **482,896** rows, leaving **5,231,112** clean flights.

### **4.5) Feature engineering:**
#### **4.5.1) Departure hour**

`SCHEDULED_DEPARTURE` stores time as `HHMM` numbers (like `830` for 08:30 or `1415` for 14:15). First we turn `2400` (midnight) into `0` so it never becomes hour `24`, then we divide by `100` to pull out the **hour of day** (`0` to `23`) as a cleaner feature for the model.

In [ ]:
df_feat = df_clean.withColumn(
    "sched_dep_clean",
    F.when(F.col("SCHEDULED_DEPARTURE") == 2400, 0).otherwise(F.col("SCHEDULED_DEPARTURE").cast("int"))
)

df_feat = df_feat.withColumn(
    "dep_hour",
    (F.col("sched_dep_clean") / 100).cast("int")
)

# Check that dep_hour sits cleanly between 0 and 23
df_feat.select("SCHEDULED_DEPARTURE", "sched_dep_clean", "dep_hour").show(5)
df_feat.select(F.min("dep_hour").alias("min_hour"), F.max("dep_hour").alias("max_hour")).show()

#### **4.5.2) Create new features and the target label**
Right after normalizing `2400` to `0` and pulling `dep_hour` (`0` to `23`), we add two practical features a model can use easily:
- `is_weekend`: `1` on Saturday/Sunday (`DAY_OF_WEEK` in `6, 7`), `0` on weekdays.
- `sched_speed_mph`: scheduled speed (`DISTANCE / (SCHEDULED_TIME / 60)`) as a quick flight-profile ratio.

We also create `label` (`1` if `ARRIVAL_DELAY > 15` minutes, `0` otherwise) so the target column is ready for classification.

In [ ]:
df_feat = df_clean.withColumn(
    "clean_dep_time",
    F.when(F.col("SCHEDULED_DEPARTURE") == 2400, 0).otherwise(F.col("SCHEDULED_DEPARTURE").cast("int"))
).withColumn(
    "dep_hour",
    (F.col("clean_dep_time") / 100).cast("int")
)

df_feat = df_feat.withColumn(
    "is_weekend",
    F.when(F.col("DAY_OF_WEEK").isin(6, 7), 1).otherwise(0)
).withColumn(
    "sched_speed_mph",
    F.col("DISTANCE") / (F.col("SCHEDULED_TIME") / 60.0)
)

df_feat = df_feat.withColumn(
    "label",
    F.when(F.col("ARRIVAL_DELAY") > 15, 1.0).otherwise(0.0)
)

df_feat.select("dep_hour", "is_weekend", "sched_speed_mph", "ARRIVAL_DELAY", "label").show(5)

#### **4.5.3) Encode text/categorical columns**
Spark ML models expect numbers rather than raw text. We first convert `AIRLINE`, `ORIGIN_AIRPORT`, and `DESTINATION_AIRPORT` into numeric indices using `StringIndexer`, then turn those indices into one-hot vectors using `OneHotEncoder`.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

cat_cols = ["AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
idx_cols = ["airline_idx", "origin_idx", "dest_idx"]
ohe_cols = ["airline_ohe", "origin_ohe", "dest_ohe"]

indexer = StringIndexer(
    inputCols=cat_cols,
    outputCols=idx_cols,
    handleInvalid="keep"
)
df_indexed = indexer.fit(df_feat).transform(df_feat)

encoder = OneHotEncoder(
    inputCols=idx_cols,
    outputCols=ohe_cols
)
df_encoded = encoder.fit(df_indexed).transform(df_indexed)

df_encoded.select("AIRLINE", "airline_idx", "airline_ohe", "label").show(5, truncate=False)

Each categorical column now has a clean index (`*_idx`) and one-hot encoded vector (`*_ohe`) without manually writing messy mapping rules.

#### ** 4.5.4) Assemble `features` vector and scale**
Next, we combine our numeric columns, engineered columns, and one-hot encoded categories into a single `features` vector using `VectorAssembler`. Also `StandardScaler` puts features on a comparable scale (for random forest)

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

num_cols = [
    "MONTH", "DAY", "DAY_OF_WEEK",
    "SCHEDULED_TIME", "DISTANCE",
    "dep_hour", "is_weekend", "sched_speed_mph"
]

feature_inputs = num_cols + ohe_cols

assembler = VectorAssembler(
    inputCols=feature_inputs,
    outputCol="raw_features"
)
df_assembled = assembler.transform(df_encoded)

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=False
)
df_ready = scaler.fit(df_assembled).transform(df_assembled).select("features", "label")

df_ready.show(5, truncate=False)

`df_ready` now contains both `features` (vector) and `label` (target), so feature engineering is complete and we're ready to split the data and train our model.

In [ ]:
# import matplotlib.pyplot as plt

# pdf = eda_sample.select("AIRLINE", "DAY_OF_WEEK", "CANCELLED", "ARRIVAL_DELAY").toPandas()

# # Chart 1: Average delay per airline
# delay_by_airline = pdf.groupby("AIRLINE")["ARRIVAL_DELAY"].mean().sort_values(ascending=False)
# plt.figure(figsize=(10, 5))
# plt.bar(delay_by_airline.index, delay_by_airline.values)
# plt.title("Average arrival delay per airline")
# plt.ylabel("Minutes")
# plt.xticks(rotation=45)
# plt.show()

# # Chart 2: Cancellation rate per day of week
# cancel_by_day = pdf.groupby("DAY_OF_WEEK")["CANCELLED"].mean()
# plt.figure(figsize=(10, 5))
# plt.bar(cancel_by_day.index.astype(str), cancel_by_day.values)
# plt.title("Cancellation rate per day of week (1=Monday ... 7=Sunday)")
# plt.ylabel("Cancellation rate")
# plt.show()

In [ ]:
# import matplotlib.pyplot as plt

# pdf = eda_sample.select(
#     "AIRLINE", "DAY_OF_WEEK", "CANCELLED", "ARRIVAL_DELAY"
# ).toPandas()

# pdf["CANCELLED"] = pdf["CANCELLED"].astype(float)
# pdf["ARRIVAL_DELAY"] = pdf["ARRIVAL_DELAY"].astype(float)

# delay_by_airline = pdf.groupby("AIRLINE")["ARRIVAL_DELAY"].mean().sort_values(ascending=False)
# plt.figure(figsize=(10, 5))
# plt.bar(delay_by_airline.index, delay_by_airline.values)
# plt.title("Average arrival delay per airline")
# plt.ylabel("Minutes")
# plt.xticks(rotation=45)
# plt.show()

In [ ]:
# cancel_by_day = pdf.groupby("DAY_OF_WEEK")["CANCELLED"].mean()
# plt.figure(figsize=(10, 5))
# plt.bar(cancel_by_day.index.astype(str), cancel_by_day.values)
# plt.title("Cancellation rate per day of week (1=Monday ... 7=Sunday)")
# plt.ylabel("Cancellation rate")
# plt.show()

In [ ]:
# cols = [
#     "MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
#     "SCHEDULED_DEPARTURE", "SCHEDULED_TIME", "DISTANCE",
#     "ARRIVAL_DELAY", "CANCELLED", "DIVERTED", "CANCELLATION_REASON"
# ]
# df = raw.select(cols)

# num_cols = [
#     "MONTH", "DAY_OF_WEEK", "SCHEDULED_DEPARTURE", "SCHEDULED_TIME",
#     "DISTANCE", "ARRIVAL_DELAY", "CANCELLED", "DIVERTED"
# ]
# for c in num_cols:
#     df = df.withColumn(c, F.col(c).cast("double"))

# df = df.sample(fraction=0.05, seed=42).cache()   # fast dev sample, remove for the final run

# print("Rows kept:", df.count())
# df.printSchema()

<!-- ## 4) Cleaning the data

Here we remove rows we cannot use, and fix two small bugs in the dataset.

Bug 1: some ORIGIN_AIRPORT and DESTINATION_AIRPORT values are numbers
instead of airport codes like "LAX". This is a known issue people found
on Kaggle. We remove these rows, since we cannot use them properly.

Bug 2: the time 2400 should mean midnight (0000), but it is written as
2400 in the file. We fix it so our later calculation of the hour of day
is correct. -->

In [ ]:
# n0 = df.count()

# clean = (df
#     .filter((F.col("CANCELLED") == 0) & (F.col("DIVERTED") == 0))
#     .filter(~F.col("ORIGIN_AIRPORT").rlike("^[0-9]+$") & ~F.col("DESTINATION_AIRPORT").rlike("^[0-9]+$"))
#     .dropna(subset=["ARRIVAL_DELAY", "SCHEDULED_TIME", "DISTANCE", "SCHEDULED_DEPARTURE"])
#     .dropDuplicates()
# )

# n1 = clean.count()
# print(f"{n0} -> {n1} rows ({n0 - n1} removed)")

In [ ]:
# # Fix the 2400 bug, then get the hour of day (example: 1250 -> 12)
# clean = clean.withColumn(
#     "SCHEDULED_DEPARTURE",
#     F.when(F.col("SCHEDULED_DEPARTURE") == 2400, 0).otherwise(F.col("SCHEDULED_DEPARTURE"))
# ).withColumn(
#     "dep_hour",
#     F.floor(F.col("SCHEDULED_DEPARTURE") / 100)
# )

## 5) Build the label (what we want to predict)

We turn ARRIVAL_DELAY into a simple yes/no answer.
1 means the flight was delayed more than 15 minutes.
0 means it was not.

In [ ]:
# clean = clean.withColumn("label", (F.col("ARRIVAL_DELAY") > 15).cast("int"))
# clean.groupBy("label").count().show()

## 7) Feature Engineering
Turn columns into features the model can use

Airports have too many different values (around 320). If we encode all
of them, we get too many columns, and the model can break. So first we
keep only the top 25 busiest airports, and put every other airport into
one group called "OTHER".

Then we use StringIndexer and OneHotEncoder, as explained in Part 4, to
turn AIRLINE, ORIGIN_AIRPORT and DESTINATION_AIRPORT into numbers.

Finally we use VectorAssembler to put all our chosen features together
into one single column called "features". This is the format the model
expects.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

top_origins = [r["ORIGIN_AIRPORT"] for r in
               clean.groupBy("ORIGIN_AIRPORT").count().orderBy(F.desc("count")).limit(25).collect()]
top_dests = [r["DESTINATION_AIRPORT"] for r in
             clean.groupBy("DESTINATION_AIRPORT").count().orderBy(F.desc("count")).limit(25).collect()]

clean = (clean
    .withColumn("origin_bucket", F.when(F.col("ORIGIN_AIRPORT").isin(top_origins), F.col("ORIGIN_AIRPORT")).otherwise("OTHER"))
    .withColumn("dest_bucket", F.when(F.col("DESTINATION_AIRPORT").isin(top_dests), F.col("DESTINATION_AIRPORT")).otherwise("OTHER"))
)

idx_airline = StringIndexer(inputCol="AIRLINE", outputCol="airline_idx", handleInvalid="keep")
idx_origin  = StringIndexer(inputCol="origin_bucket", outputCol="origin_idx", handleInvalid="keep")
idx_dest    = StringIndexer(inputCol="dest_bucket", outputCol="dest_idx", handleInvalid="keep")

ohe = OneHotEncoder(
    inputCols=["airline_idx", "origin_idx", "dest_idx"],
    outputCols=["airline_vec", "origin_vec", "dest_vec"]
)

asm = VectorAssembler(
    inputCols=["airline_vec", "origin_vec", "dest_vec", "MONTH", "DAY_OF_WEEK", "dep_hour", "SCHEDULED_TIME", "DISTANCE"],
    outputCol="features"
)

## 8) Train two models and compare them

We split the data into a training part (80%) and a test part (20%).
We always check the score on the test part only, never on the training
part. The model has never seen the test part before, so this score tells
us how well it would do on new, real flights.

We train two models: Logistic Regression and Random Forest.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

train, test = clean.randomSplit([0.8, 0.2], seed=42)

models = {
    "LogisticRegression": LogisticRegression(featuresCol="features", labelCol="label"),
    "RandomForest": RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=8, seed=42)
}

f1  = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

results = []
for name, clf in models.items():
    pipeline = Pipeline(stages=[idx_airline, idx_origin, idx_dest, ohe, asm, clf])
    pred = pipeline.fit(train).transform(test)
    row = (name, acc.evaluate(pred), f1.evaluate(pred), auc.evaluate(pred))
    results.append(row)
    print(name, "| accuracy:", round(row[1], 3), "| f1:", round(row[2], 3), "| auc:", round(row[3], 3))

## Saving the results to Streamlit

We save small tables as CSV files. Streamlit will read these files later
to build the dashboard. Streamlit does not need Spark or HDFS, just these
small files.

In [ ]:
# import os

# os.makedirs("outputs", exist_ok=True)

# stats_airline = (clean.groupBy("AIRLINE")
#     .agg(
#         F.count("*").alias("flights"),
#         F.round(F.avg("ARRIVAL_DELAY"), 2).alias("avg_arr_delay"),
#         F.round(F.avg("label") * 100, 1).alias("delay_rate_pct")
#     )
#     .orderBy(F.desc("avg_arr_delay"))
# )
# stats_airline.toPandas().to_csv("outputs/stats_airline.csv", index=False)

# stats_hour = (clean.groupBy("dep_hour")
#     .agg(
#         F.count("*").alias("flights"),
#         F.round(F.avg("ARRIVAL_DELAY"), 2).alias("avg_arr_delay")
#     )
#     .orderBy("dep_hour")
# )
# stats_hour.toPandas().to_csv("outputs/stats_hour.csv", index=False)

# pd.DataFrame(results, columns=["model", "accuracy", "f1", "auc"]).to_csv("outputs/model_scores.csv", index=False)

# print("Files created:", os.listdir("outputs"))

## 3.2) Feature Selection & Data Leakage Verification

### Initial Manual Exploration
After reviewing the dataset schema and domain meaning of all 31 columns, we separated the features into two groups:

* **Safe Features (Known before takeoff):** `MONTH`, `DAY_OF_WEEK`, `AIRLINE`, `ORIGIN_AIRPORT`, `DESTINATION_AIRPORT`, `SCHEDULED_DEPARTURE`, `SCHEDULED_TIME`, `DISTANCE`. *(Note: `TAIL_NUMBER`, `FLIGHT_NUMBER`, and `DAY` are excluded to prevent overfitting/high cardinality, and `SCHEDULED_ARRIVAL` is redundant with departure + duration).*
* **Dropped Columns (Post-takeoff / Data Leakage):** `DEPARTURE_TIME`, `DEPARTURE_DELAY`, `TAXI_OUT`, `WHEELS_OFF`, `WHEELS_ON`, `TAXI_IN`, `ARRIVAL_TIME`, `ELAPSED_TIME`, `AIR_TIME`, `CANCELLATION_REASON`, and the 5 specific delay breakdown columns (`AIR_SYSTEM_DELAY`, `SECURITY_DELAY`, `AIRLINE_DELAY`, `LATE_AIRCRAFT_DELAY`, `WEATHER_DELAY`). Including these would leak future information that does not exist when predicting a delay prior to departure. *(Note: `CANCELLED`, `DIVERTED`, and `ARRIVAL_DELAY` are kept temporarily only to filter invalid flights and create our target label, then removed before training).*

### Proving Data Leakage in the Delay Columns
According to the dataset documentation, the 5 specific delay columns are only populated when a flight is delayed by 15 minutes or more, and **their sum equals the exact `ARRIVAL_DELAY`**. To prove this empirically before dropping them, we sum those 5 columns together and compare the result side-by-side against `ARRIVAL_DELAY`.

## Data Quality Checks (Known Dataset Anomalies)

Based on the Kaggle dataset documentation, two known formatting quirks exist in the 2015 flight records:
1. **Numeric Airport Codes:** In October 2015, the DOT temporarily replaced 3-letter IATA airport codes (e.g., `JFK`) with 5-digit numeric IDs (e.g., `12478`).
2. **Midnight as `2400`:** Some midnight departures are encoded as `2400` instead of `0000`, which would create a invalid hour `24` when extracting departure hours.

We check whether both anomalies appear in our sample before cleaning them.

In [ ]:
# 1. Check for numeric airport codes (if text turns into a valid integer, it's numeric)
num_origin = eda_sample.filter(F.col("ORIGIN_AIRPORT").cast("int").isNotNull()).count()
num_dest   = eda_sample.filter(F.col("DESTINATION_AIRPORT").cast("int").isNotNull()).count()

# midnight flights written as 2400
midnight_2400 = eda_sample.filter(F.col("SCHEDULED_DEPARTURE") == "2400").count()

print("Numeric airports -> Origin:", num_origin, "| Dest:", num_dest)
print("Midnight 2400 departures:", midnight_2400)

In [ ]:
# 1. Retard moyen par compagnie (trié du pire au meilleur)
eda_sample.groupBy("AIRLINE").agg(
    F.count("*").alias("nb_vols"),
    F.round(F.avg(F.col("ARRIVAL_DELAY").cast("double")), 2).alias("retard_moyen_min")
).orderBy(F.desc("retard_moyen_min")).show()

# 2. Taux d'annulation par jour de la semaine (1=Lundi ... 7=Dimanche)
eda_sample.groupBy("DAY_OF_WEEK").agg(
    F.count("*").alias("nb_vols"),
    F.round(F.avg(F.col("CANCELLED").cast("double")) * 100, 2).alias("taux_annulation_pct")
).orderBy("DAY_OF_WEEK").show()

In [ ]:
import matplotlib.pyplot as plt

pdf = eda_sample.select(
    "AIRLINE", "DAY_OF_WEEK", "MONTH", "CANCELLED", "ARRIVAL_DELAY"
).toPandas()

pdf["CANCELLED"] = pdf["CANCELLED"].astype(float)
pdf["ARRIVAL_DELAY"] = pdf["ARRIVAL_DELAY"].astype(float)

# Graphique 1 : Retard moyen par compagnie
delay_by_airline = pdf.groupby("AIRLINE")["ARRIVAL_DELAY"].mean().sort_values(ascending=False)
plt.figure(figsize=(10, 5))
plt.bar(delay_by_airline.index, delay_by_airline.values)
plt.title("Retard moyen à l'arrivée par compagnie")
plt.ylabel("Minutes")
plt.xticks(rotation=45)
plt.show()

# Graphique 2 : Taux d'annulation par jour de la semaine
cancel_by_day = pdf.groupby("DAY_OF_WEEK")["CANCELLED"].mean()
plt.figure(figsize=(10, 5))
plt.bar(cancel_by_day.index.astype(str), cancel_by_day.values)
plt.title("Taux d'annulation par jour de la semaine (1=Lundi ... 7=Dimanche)")
plt.ylabel("Taux d'annulation")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

pdf = eda_sample.select(
    "AIRLINE", "DAY_OF_WEEK", "MONTH", "CANCELLED", "ARRIVAL_DELAY"
).toPandas()

pdf["CANCELLED"] = pdf["CANCELLED"].astype(float)
pdf["ARRIVAL_DELAY"] = pdf["ARRIVAL_DELAY"].astype(float)

# Graphique 1 : Retard moyen par compagnie
delay_by_airline = pdf.groupby("AIRLINE")["ARRIVAL_DELAY"].mean().sort_values(ascending=False)
plt.figure(figsize=(10, 5))
plt.bar(delay_by_airline.index, delay_by_airline.values)
plt.title("Retard moyen à l'arrivée par compagnie")
plt.ylabel("Minutes")
plt.xticks(rotation=45)
plt.show()

# Graphique 2 : Taux d'annulation par jour de la semaine
cancel_by_day = pdf.groupby("DAY_OF_WEEK")["CANCELLED"].mean()
plt.figure(figsize=(10, 5))
plt.bar(cancel_by_day.index.astype(str), cancel_by_day.values)
plt.title("Taux d'annulation par jour de la semaine (1=Lundi ... 7=Dimanche)")
plt.ylabel("Taux d'annulation")
plt.show()

In [ ]:
cols = [
    "MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
    "SCHEDULED_DEPARTURE", "SCHEDULED_TIME", "DISTANCE",
    "ARRIVAL_DELAY", "CANCELLED", "DIVERTED", "CANCELLATION_REASON"
]
df = raw.select(cols)

num_cols = [
    "MONTH", "DAY_OF_WEEK", "SCHEDULED_DEPARTURE", "SCHEDULED_TIME",
    "DISTANCE", "ARRIVAL_DELAY", "CANCELLED", "DIVERTED"
]
for c in num_cols:
    df = df.withColumn(c, F.col(c).cast("double"))

# Échantillon de développement rapide (retirer .sample(fraction=0.05, seed=42) pour le run final 100%)
df = df.sample(fraction=0.05, seed=42).cache()

print("Lignes conservées :", df.count())
df.printSchema()

In [ ]:
n0 = df.count()

clean = (df
    .filter((F.col("CANCELLED") == 0) & (F.col("DIVERTED") == 0))
    .filter(~F.col("ORIGIN_AIRPORT").rlike("^[0-9]+$") & ~F.col("DESTINATION_AIRPORT").rlike("^[0-9]+$"))
    .dropna(subset=["ARRIVAL_DELAY", "SCHEDULED_TIME", "DISTANCE", "SCHEDULED_DEPARTURE"])
    .dropDuplicates()
)

# Remplacer 2400 (minuit) par 0 et extraire l'heure de départ (ex: 1250 -> 12)
clean = clean.withColumn(
    "SCHEDULED_DEPARTURE",
    F.when(F.col("SCHEDULED_DEPARTURE") == 2400, 0).otherwise(F.col("SCHEDULED_DEPARTURE"))
).withColumn(
    "dep_hour",
    F.floor(F.col("SCHEDULED_DEPARTURE") / 100)
)

n1 = clean.count()
print(f"{n0} -> {n1} rows ({n0 - n1} removed)")

In [ ]:
clean = clean.withColumn("label", (F.col("ARRIVAL_DELAY") > 15).cast("int"))
clean.groupBy("label").count().show()

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

top_origins = [r["ORIGIN_AIRPORT"] for r in 
               clean.groupBy("ORIGIN_AIRPORT").count().orderBy(F.desc("count")).limit(25).collect()]
top_dests = [r["DESTINATION_AIRPORT"] for r in 
             clean.groupBy("DESTINATION_AIRPORT").count().orderBy(F.desc("count")).limit(25).collect()]

clean = (clean
    .withColumn("origin_bucket", F.when(F.col("ORIGIN_AIRPORT").isin(top_origins), F.col("ORIGIN_AIRPORT")).otherwise("OTHER"))
    .withColumn("dest_bucket", F.when(F.col("DESTINATION_AIRPORT").isin(top_dests), F.col("DESTINATION_AIRPORT")).otherwise("OTHER"))
)

idx_airline = StringIndexer(inputCol="AIRLINE", outputCol="airline_idx", handleInvalid="keep")
idx_origin  = StringIndexer(inputCol="origin_bucket", outputCol="origin_idx", handleInvalid="keep")
idx_dest    = StringIndexer(inputCol="dest_bucket", outputCol="dest_idx", handleInvalid="keep")

ohe = OneHotEncoder(
    inputCols=["airline_idx", "origin_idx", "dest_idx"],
    outputCols=["airline_vec", "origin_vec", "dest_vec"]
)

asm = VectorAssembler(
    inputCols=["airline_vec", "origin_vec", "dest_vec", "MONTH", "DAY_OF_WEEK", "dep_hour", "SCHEDULED_TIME", "DISTANCE"],
    outputCol="features"
)

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

train, test = clean.randomSplit([0.8, 0.2], seed=42)

models = {
    "LogisticRegression": LogisticRegression(featuresCol="features", labelCol="label"),
    "RandomForest": RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=8, seed=42)
}

f1  = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

results = []
for name, clf in models.items():
    pipeline = Pipeline(stages=[idx_airline, idx_origin, idx_dest, ohe, asm, clf])
    pred = pipeline.fit(train).transform(test)
    row = (name, acc.evaluate(pred), f1.evaluate(pred), auc.evaluate(pred))
    results.append(row)
    print(name, "| accuracy:", round(row[1], 3), "| f1:", round(row[2], 3), "| auc:", round(row[3], 3))

In [ ]:
import os
import pandas as pd

os.makedirs("outputs", exist_ok=True)

# 1. Stats par compagnie
stats_airline = (clean.groupBy("AIRLINE")
    .agg(
        F.count("*").alias("flights"),
        F.round(F.avg("ARRIVAL_DELAY"), 2).alias("avg_arr_delay"),
        F.round(F.avg("label") * 100, 1).alias("delay_rate_pct")
    )
    .orderBy(F.desc("avg_arr_delay"))
)
stats_airline.toPandas().to_csv("outputs/stats_airline.csv", index=False)

# 2. Stats par heure de départ
stats_hour = (clean.groupBy("dep_hour")
    .agg(
        F.count("*").alias("flights"),
        F.round(F.avg("ARRIVAL_DELAY"), 2).alias("avg_arr_delay")
    )
    .orderBy("dep_hour")
)
stats_hour.toPandas().to_csv("outputs/stats_hour.csv", index=False)

# 3. Scores des modèles ML
pd.DataFrame(results, columns=["model", "accuracy", "f1", "auc"]).to_csv("outputs/model_scores.csv", index=False)

print("Fichiers générés :", os.listdir("outputs"))

## Selection des colonnes utiles

In [ ]:
cols = ["MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
        "SCHEDULED_DEPARTURE", "SCHEDULED_TIME", "DISTANCE",
        "ARRIVAL_DELAY", "CANCELLED", "DIVERTED", "CANCELLATION_REASON"]
clean = df.select(cols)

num_cols = ["MONTH", "DAY_OF_WEEK", "SCHEDULED_DEPARTURE", "SCHEDULED_TIME",
            "DISTANCE", "ARRIVAL_DELAY", "CANCELLED", "DIVERTED"]
for c in num_cols:
    clean = clean.withColumn(c, F.col(c).cast("double"))

clean = clean.sample(fraction=0.05, seed=42).cache()   # 5% while developing, remove .sample() for the final run
print(clean.count())
clean.printSchema()

## EDA

In [ ]:
clean.describe(["ARRIVAL_DELAY", "DISTANCE", "SCHEDULED_TIME"]).show()

In [ ]:
clean.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in clean.columns]).show()

## delay and volume per airline

In [ ]:
stats_airline = (clean.groupBy("AIRLINE")
                   .agg(F.count("*").alias("flights"),
                        F.round(F.avg("ARRIVAL_DELAY"), 2).alias("avg_arr_delay"))
                   .orderBy(F.desc("avg_arr_delay")))
stats_airline.show()

## Cancellations

In [ ]:
cancel_by_airline = (clean.groupBy("AIRLINE")
                       .agg(F.count("*").alias("flights"),
                            F.round(F.avg("CANCELLED") * 100, 2).alias("cancel_rate_pct"))
                       .orderBy(F.desc("cancel_rate_pct")))
cancel_by_airline.show()

cancel_reasons = (clean.filter(F.col("CANCELLED") == 1)
                    .groupBy("CANCELLATION_REASON").count()
                    .orderBy(F.desc("count")))
cancel_reasons.show()
# A = Airline/Carrier, B = Weather, C = National Air System, D = Security

## Nettoyage

In [ ]:
n0 = clean.count()

clean2 = (clean
    .filter((F.col("CANCELLED") == 0) & (F.col("DIVERTED") == 0))
    .filter(~F.col("ORIGIN_AIRPORT").rlike("^[0-9]+$") & ~F.col("DESTINATION_AIRPORT").rlike("^[0-9]+$"))
    .dropna(subset=["ARRIVAL_DELAY", "SCHEDULED_TIME", "DISTANCE", "SCHEDULED_DEPARTURE"])
    .dropDuplicates())

n1 = clean2.count()
print(f"{n0} -> {n1} rows ({n0 - n1} removed)")

In [ ]:
top_origins = [r["ORIGIN_AIRPORT"] for r in
               clean2.groupBy("ORIGIN_AIRPORT").count().orderBy(F.desc("count")).limit(25).collect()]
top_dests = [r["DESTINATION_AIRPORT"] for r in
             clean2.groupBy("DESTINATION_AIRPORT").count().orderBy(F.desc("count")).limit(25).collect()]

clean2 = (clean2
    .withColumn("origin_bucket", F.when(F.col("ORIGIN_AIRPORT").isin(top_origins), F.col("ORIGIN_AIRPORT")).otherwise("OTHER"))
    .withColumn("dest_bucket", F.when(F.col("DESTINATION_AIRPORT").isin(top_dests), F.col("DESTINATION_AIRPORT")).otherwise("OTHER")))

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

idx_airline = StringIndexer(inputCol="AIRLINE", outputCol="airline_idx", handleInvalid="keep")
idx_origin  = StringIndexer(inputCol="origin_bucket", outputCol="origin_idx", handleInvalid="keep")
idx_dest    = StringIndexer(inputCol="dest_bucket", outputCol="dest_idx", handleInvalid="keep")

ohe = OneHotEncoder(inputCols=["airline_idx", "origin_idx", "dest_idx"],
                     outputCols=["airline_vec", "origin_vec", "dest_vec"])

asm = VectorAssembler(inputCols=["airline_vec", "origin_vec", "dest_vec",
                                  "MONTH", "DAY_OF_WEEK",
                                  "SCHEDULED_TIME", "DISTANCE"],
                       outputCol="features")

In [ ]:
clean2 = clean2.withColumn("label", (F.col("ARRIVAL_DELAY") > 15).cast("int"))
clean2.groupBy("label").count().show()

## Machine learning  entraînement et évaluation de deux modèles

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

train, test = clean2.randomSplit([0.8, 0.2], seed=42)

models = {
    "LogisticRegression": LogisticRegression(featuresCol="features", labelCol="label"),
    "RandomForest": RandomForestClassifier(featuresCol="features", labelCol="label",
                                           numTrees=50, maxDepth=8, seed=42)
}

f1  = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

results = []

In [ ]:
for name, clf in models.items():
    pipeline = Pipeline(stages=[idx_airline, idx_origin, idx_dest, ohe, asm, clf])
    pred = pipeline.fit(train).transform(test)
    row = (name, acc.evaluate(pred), f1.evaluate(pred), auc.evaluate(pred))
    results.append(row)
    print(name, "accuracy:", round(row[1],3), "f1:", round(row[2],3), "auc:", round(row[3],3))

In [ ]:
# df = spark.read.csv("hdfs://localhost:9000/data/raw/flights.csv", header=True)
# print(df.count())
# df.printSchema()

In [ ]:
# df.printSchema()

In [ ]:
# print(df.count())

In [ ]:
# all = ["YEAR","MONTH","DAY","DAY_OF_WEEK","FLIGHT_NUMBER","SCHEDULED_DEPARTURE",
#            "DEPARTURE_TIME","DEPARTURE_DELAY","TAXI_OUT","WHEELS_OFF","SCHEDULED_TIME",
#            "ELAPSED_TIME","AIR_TIME","DISTANCE","WHEELS_ON","TAXI_IN","SCHEDULED_ARRIVAL",
#            "ARRIVAL_TIME","ARRIVAL_DELAY","DIVERTED","CANCELLED","AIR_SYSTEM_DELAY",
#            "SECURITY_DELAY","AIRLINE_DELAY","LATE_AIRCRAFT_DELAY","WEATHER_DELAY"]
# full = raw
# for c in all:
#     full = full.withColumn(c, F.col(c).cast("double"))
# eda = full.sample(fraction=0.05, seed=42).cache()
# print(eda.count(), len(eda.columns))   # ~290k rows, 31 columns

In [ ]:
# cols = ["MONTH", "DAY", "DAY_OF_WEEK", "AIRLINE", "FLIGHT_NUMBER","TAIL_NUMBER","ORIGIN_AIRPORT","DESTINATION_AIRPORT","SCHEDULED_DEPARTURE", "SCHEDULED_TIME",
#         "DISTANCE", "SCHEDULED_ARRIVAL"]

In [ ]:
# eda = full.sample(fraction=0.05, seed=42).cache()

# print("Rows:", eda.count())
# print("Columns:", len(eda.columns))

In [ ]:
# missing = eda.select([
#     F.round(
#         F.avg(F.col(c).isNull().cast("int")) * 100, 2
#     ).alias(c)
#     for c in eda.columns
# ])

# missing.toPandas().T.sort_values(0, ascending=False)

In [ ]:
# eda.select(
#     "DEPARTURE_DELAY",
#     "ARRIVAL_DELAY",
#     "DISTANCE",
#     "AIR_TIME",
#     "SCHEDULED_TIME"
# ).describe().show()

In [ ]:

# airline_delays = (
#     eda.groupBy("AIRLINE")
#     .agg(
#         F.count("*").alias("flights"),
#         F.round(F.avg("DEPARTURE_DELAY"), 2).alias("avg_delay")
#     )
#     .orderBy(F.desc("avg_delay"))
# )

# airline_delays.show()


In [ ]:
# flights = eda.filter(
#     (F.col("CANCELLED") == 0) &
#     (F.col("DIVERTED") == 0) &
#     F.col("DEPARTURE_DELAY").isNotNull()
# )

# flights = flights.withColumn(
#     "label",
#     F.when(F.col("DEPARTURE_DELAY") >= 15, 1).otherwise(0)
# )

# flights.groupBy("label").count().show()
